# Stage 2 - Preprocessing / cleaning

YouTube comments are messy (slang, emojis, ALLCAPS, typos...). This stage turns the raw text
into something the NLP models can actually work with.

We use the two resource files from Moodle:
- `contractions.txt` -> expand can't -> cannot etc
- `acronyms.csv` -> expand lol -> laughing out loud, imo -> in my opinion ...

We keep two versions of the text:
- **clean_text**: lightly cleaned, still readable -> used later for RAG / showing examples
- **lemmas**: aggressive (lemmatized, no stopwords) -> used for keywords + topic modeling

In: `../data/raw_comments.csv`  Out: `../data/clean_comments.csv`

## Merge the two scrape files

Run this once after both people have finished notebook 1.
It deduplicates on `comment_id` so overlapping replies don't get counted twice.
Skip this cell if you already have `raw_comments.csv` in the data folder.

In [1]:
import pandas as pd, os

f0 = "../data/raw_comments_0.csv"
f1 = "../data/raw_comments_1.csv"
merged_path = "../data/raw_comments.csv"

if os.path.exists(merged_path):
    print("already merged, skipping")
else:
    parts = [pd.read_csv(f) for f in [f0, f1] if os.path.exists(f)]
    if len(parts) == 0:
        raise FileNotFoundError("run notebook 1 first (both scraper IDs)")
    df_merged = (pd.concat(parts)
                   .drop_duplicates("comment_id")
                   .reset_index(drop=True))
    df_merged.to_csv(merged_path, index=False)
    print(f"merged {len(df_merged)} unique comments from {len(parts)} file(s)")
    print(f"  -> {merged_path}")


merged 19270 unique comments from 2 file(s)
  -> ../data/raw_comments.csv


In [2]:
import pandas as pd
import re
import csv
import spacy

In [3]:
df = pd.read_csv("../data/raw_comments.csv")
# drop empty / non string just in case
df = df.dropna(subset=["text"])
df = df[df["text"].str.strip().astype(bool)].reset_index(drop=True)
print(len(df), "comments to clean")
df.head(3)

19266 comments to clean


,comment_id,parent_id,author,published_at,like_count,text,video_id,is_reply
0,UgzfpT3qlUjjDj_cbb54AaABAg,NaN,@GSMArenaOfficial,2025-03-19T23:01:19Z,8,Our Apple iOS 18.2 review is up now: https://y...,aDqzDoJPkaA,False
1,UgyAYaZ5cKRbmsZePYV4AaABAg,NaN,@cyberlynxplays,2026-04-29T13:14:02Z,0,Is Apple in the red? Because what do you mean ...,aDqzDoJPkaA,False
2,UgysYK9IbV22Ij3EMMh4AaABAg,NaN,@bernardsteele-dadzie7656,2026-04-26T15:29:29Z,0,5years of software support when sansung budget...,aDqzDoJPkaA,False


## Load the resource files

In [4]:
contractions = {}
for line in open("../data/contractions.txt", encoding="utf-8"):
    line = line.strip()
    if not line or ":" not in line:
        continue
    k, v = line.split(":", 1)
    contractions[k.lower()] = v

acronyms = {}
with open("../data/acronyms.csv", encoding="utf-8", errors="ignore") as f:
    for row in csv.reader(f):
        if len(row) >= 2 and row[0].strip():
            acronyms[row[0].strip().lower()] = row[1].strip()

print(len(contractions), "contractions,", len(acronyms), "acronyms")

125 contractions, 5346 acronyms


## Cleaning functions

Order matters: lowercase -> kill urls -> expand contractions -> expand acronyms -> strip emojis/junk.
We do acronyms token by token so trailing punctuation (like `lol,`) still matches.

In [5]:
URL_RE = re.compile(r"http\S+|www\.\S+")
# rough emoji / non latin range
EMOJI_RE = re.compile("["
    "\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "]+", flags=re.UNICODE)

def expand_contractions(text):
    for k, v in contractions.items():
        text = re.sub(r"\b" + re.escape(k) + r"\b", v, text)
    return text

def expand_acronyms(text):
    out = []
    for tok in text.split():
        # split the word from any leading/trailing punctuation
        m = re.match(r"^(\W*)(.*?)(\W*)$", tok)
        pre, core, post = m.groups()
        core = acronyms.get(core.lower(), core)
        out.append(pre + core + post)
    return " ".join(out)

def basic_clean(text):
    text = str(text).lower()
    text = URL_RE.sub(" ", text)
    text = expand_contractions(text)
    text = expand_acronyms(text)
    text = EMOJI_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [6]:
# quick test before running on everything
sample = "OMG i can't believe this lol, imo its the best fr 😂 check http://x.com"
print(basic_clean(sample))

oh my God i cannot believe this laughing out loud, in my opinion its the best fr check


In [7]:
df["clean_text"] = df["text"].apply(basic_clean)
# drop ones that became empty after cleaning
df = df[df["clean_text"].str.strip().astype(bool)].reset_index(drop=True)
df[["text", "clean_text"]].head(8)

,text,clean_text
0,Our Apple iOS 18.2 review is up now: https://y...,our apple ios 18.2 review is up now:
1,Is Apple in the red? Because what do you mean ...,is apple in the red? because what do you mean ...
2,5years of software support when sansung budget...,5years of software support when sansung budget...
3,Can someone please buy me iPhone please 🥺,can someone please buy me iphone please
4,This is such a pos. Everytime I pick it up I f...,this is such a Parent Over Shoulder. everytime...
5,I’m upgrading my iphone 6s plus to iphone 16 n...,i’m upgrading my iphone 6s plus to iphone 16 n...
6,I want to upgrade my 12 to a 16 because I’m co...,i want to upgrade my 12 to a 16 because i’m co...
7,@S@ShiB88 is good. Big upgrade for me,@s@shib88 is good. big upgrade for me


## Optional - spell correction

TextBlob can fix typos but it's SLOW (it checks every word), so running it on all 20k would take
forever. We left it off by default and only ran it on a small sample to show it works / for the report.
Flip `DO_SPELLCHECK = True` if you want it on everything (go get a coffee).

In [11]:
from textblob import TextBlob

DO_SPELLCHECK = True

def spellfix(text):
    return str(TextBlob(text).correct())

# demo on a few rows
for t in df["clean_text"].head(3):
    print("before:", t)
    print("after :", spellfix(t))
    print()

if DO_SPELLCHECK:
    df["clean_text"] = df["clean_text"].apply(spellfix)

before: our apple ios 18.2 review is up now:
after : our apple is 18.2 review is up now:

before: is apple in the red? because what do you mean my phone doesn’t come with a charger
after : is apple in the red? because what do you mean my phone doesn’t come with a charge

before: 5years of software support when sansung budgets have six years?
after : years of software support when shantung budget have six years?



## Lemmas for the topic / keyword stages

spaCy for lemmatization + dropping stopwords/punctuation. Same `en_core_web_sm` model from Lab 2.
`nlp.pipe` with batches is way faster than calling nlp() in a loop.

In [12]:
# if not installed yet:  python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

def to_lemmas(doc):
    return " ".join(
        t.lemma_ for t in doc
        if not t.is_stop and not t.is_punct and not t.is_space and len(t) > 2
    )

lemmas = []
for doc in nlp.pipe(df["clean_text"].tolist(), batch_size=200):
    lemmas.append(to_lemmas(doc))
df["lemmas"] = lemmas
df[["clean_text", "lemmas"]].head(8)

,clean_text,lemmas
0,our apple is 18.2 review is up now:,apple 18.2 review
1,is apple in the red? because what do you mean ...,apple red mean phone come charge
2,years of software support when shantung budget...,year software support shantung budget year
3,can someone please buy me phone please,buy phone
4,this is such a Parent Over Shoulder. everytime...,Parent Shoulder everytime pick feel like find ...
5,i’m upbraiding my phone is plus to phone 16 ne...,upbraid phone plus phone week instant Message ...
6,i want to upgrade my 12 to a 16 because i’m co...,want upgrade completely storage need storage like
7,@s@shib88 is good. big upgrade for me,@s@shib88 good big upgrade


In [13]:
df.to_csv("../data/clean_comments.csv", index=False)
print("saved", len(df), "cleaned rows -> ../data/clean_comments.csv")

saved 18771 cleaned rows -> ../data/clean_comments.csv
